### Objective: Apply clustering algorithms to segment an e-commerce company's customer base into distinct groups based on purchasing behaviour, enabling targeted marketing strategies.

Tech Stack: Python, pandas, scikit-learn (KMeans), matplotlib, seaborn, Jupyter Notebook 

In [ ]:
#importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [ ]:
#loading the dataset
df=pd.read_csv("data\ecommerce_customer_data.csv")

# Data Exploration

In [ ]:
#rows and columns in the dataset
df.shape

In [ ]:
#data at a glance
df.head()

In [ ]:
#columns
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().any()

In [ ]:
#number of missing values
df.isnull().sum()

In [ ]:
#handling missing values
df['Returns'].fillna(df['Returns'].mean, inplace=True)

#replacing all null values in the Returns column with the mean of the column

In [ ]:
#checking again
df.isnull().sum()

In [ ]:
#checking duplicate column names
df.columns.duplicated().any()

In [ ]:
# #checking if there are duplicate values in columns
has_duplicate_values = df.T.duplicated().any()
print(has_duplicate_values)

In [ ]:
#finding the duplicated column pair
duplicate_pairs = []

for i in range(len(df.columns)):
    for j in range(i + 1, len(df.columns)):
        if df.iloc[:, i].equals(df.iloc[:, j]):
            duplicate_pairs.append((df.columns[i], df.columns[j]))

print("Duplicate column pairs:")
for pair in duplicate_pairs:
    print(pair)

In [ ]:
#dropping one of the duplicated column
df.drop(columns=['Age'], inplace=True)

In [ ]:
df.shape

In [ ]:
df.head()

# Statistical Analysis

### Average purchase value

In [ ]:
avg_purchase_val= df['Total Purchase Amount'].mean()
print(avg_purchase_val)

### Purchase frequency

In [ ]:
df.columns  

In [ ]:
print("Unique Customers:", df['Customer ID'].nunique())

In [ ]:
purchase_frequency = (df.groupby('Customer ID').size().reset_index(name='Purchase_Frequency'))
print(purchase_frequency)

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(purchase_frequency['Purchase_Frequency'],bins=20,kde=True)

plt.title("Distribution of Purchase Frequency")
plt.xlabel("Purchase Frequency")
plt.savefig('images/Purchase_frequency.png')
plt.show()

Most customers made between 1 and 5 purchases, while only a small number of customers made more than 10 purchases. This indicates that a relatively small group of highly engaged customers contributes repeated purchases. 

In [ ]:
purchase_frequency.sort_values(by='Purchase_Frequency', ascending=False).head(10)

In [ ]:
purchase_frequency.nlargest(1, 'Purchase_Frequency')

Customer with ID=48382 has the highest purchase frquency of 17

### Customer lifetime value

In [ ]:
clv = df.groupby('Customer ID')['Total Purchase Amount'].sum().reset_index(name=('Customer_Lifetime_Value'))
print(clv.head())
print(" ")
print(clv.nlargest(1,'Customer_Lifetime_Value'))

Customer with ID= 39895 has the highest customer lifetime value

In [ ]:
top10 = clv.nlargest(10, 'Customer_Lifetime_Value')

plt.figure(figsize=(10,5))
sns.barplot(data=top10,x='Customer ID',y='Customer_Lifetime_Value')

plt.title("Top 10 Customers by Customer Lifetime Value")
plt.xticks(rotation=45)
plt.grid(True)
plt.savefig('images/CLV.png')
plt.show()

# RFM (Recency, Frequency, Monetary Analysis)

In [ ]:
#changing the data type of the Purchase date column
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])
print(df['Purchase Date'].dtype)

### Recency

In [ ]:
df.columns

In [ ]:
reference_date = df['Purchase Date'].max() + pd.Timedelta(days=1)

In [ ]:
recency=df.groupby('Customer ID')['Purchase Date'].max().reset_index()
recency['Recency'] = (reference_date - recency['Purchase Date']).dt.days

recency = recency[['Customer ID','Recency']]
recency.head()

In [ ]:
recency.sort_values(by='Recency',ascending=False).head(10)

Customer with Customer ID 10742 has highest recency

In [ ]:
recency.sort_values(by='Recency',ascending=True).head(10)

Customer with Customer ID = 22010 is the most recent customer i.e has made a purchase recently

In [ ]:
plt.figure(figsize=(7,5))
sns.histplot(recency['Recency'], bins=30, kde=True)

plt.title("Distribution of Recency")
plt.xlabel("Recency (Days)")
plt.ylabel("Number of Customers")
plt.savefig('images/recency.png')
plt.show()

From the graph it is clear that the recency has decreased over the days

### Frequency

In [ ]:
frequency = df.groupby('Customer ID').size().reset_index(name='Frequency')
frequency.sort_values(by='Frequency', ascending=False).head(1)

Customer with ID=48382 has the highest purchase frquency of 17

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(frequency['Frequency'], discrete=True)

plt.title("Distribution of Purchase Frequency")
plt.xlabel("Frequency")
plt.ylabel("Number of Customers")
plt.xticks(range(1, frequency['Frequency'].max() + 1))
plt.savefig('images/pur_frq_distribution.png')
plt.show()

The maximum purchase frequency observed was 17 purchases. However, the majority of customers made between 1 and 5 purchases, indicating that repeat purchases beyond five transactions are relatively uncommon.

### Monetary

In [ ]:
df.columns

In [ ]:
monetary=df.groupby('Customer ID')['Total Purchase Amount'].sum().reset_index(name='Monetary')
monetary.sort_values(by='Monetary', ascending=False).head(1)

Customer with ID = 39895 has the highest contribution to Total amount 

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(monetary['Monetary'],bins=30,kde=True)
plt.title("Monetary Distribution")
plt.savefig('images/monetary_distribution.png')
plt.show()

The maximum amount observed was 50659. However, the majority of customers made sales between 10,000 and 25,000, indicating that repeat sales beyond 25,000 is relatively uncommon."


### Merging all of RFM

In [ ]:
rfm = recency.merge(frequency,on='Customer ID')
rfm = rfm.merge(monetary,on='Customer ID')
rfm.tail(10)


In [ ]:
sns.pairplot(rfm[['Recency','Frequency','Monetary']])
plt.savefig('images/rfm_pairplot.png')
plt.show()

# Standardisation

In [ ]:
rfm.columns

In [ ]:
rfm_features = rfm[['Recency','Frequency','Monetary']]
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

Standardization puts all three features on the same scale so that Recency, Frequency, and Monetary contribute equally to the clustering

In [ ]:
#converting back to dataframe 
rfm_scaled = pd.DataFrame(rfm_scaled,columns=rfm_features.columns)
rfm_scaled.head()

In [ ]:
rfm_scaled.describe().round(2)

### Applying elbow method on this scaled data

In [ ]:
wcss = []
cluster_centre=[]
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k,random_state=42,n_init=10)
    kmeans.fit(rfm_scaled)
    wcss.append(kmeans.inertia_)
cluster_centre.append(kmeans.cluster_centers_)

In [ ]:
cluster_centre[:5]

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(range(1, 11), wcss, marker='o', linewidth=2)

plt.title("Elbow Method for Optimal Number of Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Within-Cluster Sum of Squares (WCSS)")

plt.xticks(range(1, 11))
plt.grid(alpha=0.3)
plt.savefig('images/elbow_method.png')
plt.show()

The Elbow Method was used to determine the optimal number of clusters. The Within-Cluster Sum of Squares (WCSS) decreases as the number of clusters increases. The point where the reduction in WCSS begins to level off (the "elbow") was selected as the optimal value of K for K-Means clustering.

Here, The optimal value of k is taken as 3 

# Plottig

In [ ]:
df.columns

In [ ]:
#adding cluster labels
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3,random_state=42,n_init=10)

rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)
rfm.head()

### Scatter plot between frequency and monetary

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(data=rfm,x='Frequency',y='Monetary',hue='Cluster',palette='Set2',)

plt.title("Customer Segments: Frequency vs Monetary")
plt.xlabel("Purchase Frequency")
plt.ylabel("Monetary Value")
plt.legend(title="Cluster")
plt.savefig('images/freq_vs_monetary.png')
plt.show()

- Top-right → Loyal high-value customers
- Bottom-left → Low-value customers
- Other regions → Medium-value segments

### Scatter plot between recency and monetary

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(data=rfm,x='Frequency',y='Recency',hue='Cluster',palette='Set2',)

plt.title("Customer Segments: Frequency vs Monetary")
plt.xlabel("Customer Recency")
plt.ylabel("Monetary Value")
plt.legend(title="Cluster")
plt.savefig('images/recency_vs_monetary.png')
plt.show()

- Low Recency + High Monetary → Recent high spenders (excellent customers)
- High Recency + Low Monetary → Dormant customers

### Customer profile

In [ ]:
cluster_profile = (rfm.groupby('Cluster').agg({'Recency': 'mean','Frequency': 'mean','Monetary': 'mean'}).round(2))

print(cluster_profile)

In [ ]:
sns.countplot(data=rfm, x='Cluster', hue='Cluster', palette='Set2',legend=False)
plt.title("Number of Customers per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")
plt.savefig('images/Customer_per_cluster.png')
plt.show()

| Cluster | Customer Type        | Characteristics                                     | Marketing Action                                             |
| ------: | -------------------- | --------------------------------------------------- | ------------------------------------------------------------ |
|       0 | VIP Customers        | Recent purchases, frequent buyers, highest spending | Reward with loyalty programs, exclusive offers, early access |
|       2 | At-Risk Customers    | Inactive, infrequent purchases, low spending        | Win-back campaigns, personalized discounts, reminder emails  |
|       1 | Potential Loyalists  | Moderate activity and spending                      | Cross-selling, personalized recommendations, reward points   |
